In [ ]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import os
import glob
import zipfile
import cv2
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


# ==========================================
# 2. EXTRACT ZIP DATASET
# ==========================================

zip_path = "archive (8).zip"
extract_path = "sign_dataset"

if not os.path.exists(extract_path):

    print("Extracting dataset...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("Dataset Ready")


# ==========================================
# 3. FIND DATASET FOLDER
# ==========================================

dataset_folder = None

for root, dirs, files in os.walk(extract_path):
    for d in dirs:
        if "SIGN" in d.upper():
            dataset_folder = os.path.join(root, d)

print("Dataset Folder:", dataset_folder)


# ==========================================
# 4. FRAME EXTRACTION FUNCTION
# ==========================================

def extract_frames(video_path, size=(224,224)):

    cap = cv2.VideoCapture(video_path)

    frames = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.resize(frame, size)
        frame = frame / 255.0

        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        return None

    return np.mean(frames, axis=0)


# ==========================================
# 5. LOAD VIDEOS
# ==========================================

video_files = glob.glob(dataset_folder + "/*.mp4")

print("Videos found:", len(video_files))

X = []
y = []

for video_path in video_files:

    label = os.path.basename(video_path).replace(".mp4","")

    features = extract_frames(video_path)

    if features is not None:

        # DATA AUGMENTATION (duplicate samples)
        for i in range(5):

            X.append(features)
            y.append(label)

X = np.array(X)
y = np.array(y)

print("Dataset Shape:", X.shape)


# ==========================================
# 6. LABEL ENCODING
# ==========================================

encoder = LabelEncoder()
y = encoder.fit_transform(y)


# ==========================================
# 7. TRAIN TEST SPLIT
# ==========================================

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))


# ==========================================
# 8. MACHINE LEARNING MODEL (KNN)
# ==========================================

X_train_flat = X_train.reshape(len(X_train),-1)
X_test_flat = X_test.reshape(len(X_test),-1)

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_flat,y_train)

pred_knn = knn.predict(X_test_flat)

print("KNN Accuracy:",accuracy_score(y_test,pred_knn))


# ==========================================
# 9. DEEP LEARNING MODEL
# ==========================================

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze pretrained layers
for layer in base_model.layers:
    layer.trainable = False


x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256,activation='relu')(x)
x = Dropout(0.5)(x)

predictions = Dense(len(np.unique(y)),activation='softmax')(x)

model = Model(inputs=base_model.input,outputs=predictions)


# ==========================================
# 10. COMPILE MODEL
# ==========================================

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# ==========================================
# 11. TRAIN MODEL
# ==========================================

model.fit(
    X_train,
    y_train,
    validation_data=(X_test,y_test),
    epochs=20,
    batch_size=8
)


# ==========================================
# 12. EVALUATE MODEL
# ==========================================

loss,acc = model.evaluate(X_test,y_test)

print("Deep Learning Accuracy:",acc)


# ==========================================
# 13. SAVE MODEL
# ==========================================

model.save("sign_language_model.keras")

print("Model Saved Successfully")

Extracting dataset...
Dataset Ready
Dataset Folder: sign_dataset/INDIAN SIGN LANGUAGE ANIMATED VIDEOS 
Videos found: 151
Dataset Shape: (755, 224, 224, 3)
Training samples: 604
KNN Accuracy: 0.9735099337748344
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 30s 348ms/step - accuracy: 0.0017 - loss: 5.2869 - val_accuracy: 0.0199 - val_loss: 4.9445
Epoch 2/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 30s 402ms/step - accuracy: 0.0199 - loss: 4.9741 - val_accuracy: 0.0464 - val_loss: 4.8328
Epoch 3/20
 8/76 ━━━━━━━━━━━━━━━━━━━━ 16s 240ms/step - accuracy: 0.0061 - loss: 4.8965

In [ ]:
# ==========================================
# INSTALL LIBRARIES
# ==========================================

!pip install gradio opencv-python tensorflow numpy -q


# ==========================================
# IMPORT LIBRARIES
# ==========================================

import cv2
import numpy as np
import tensorflow as tf
import gradio as gr


# ==========================================
# LOAD TRAINED MODEL
# ==========================================

model = tf.keras.models.load_model("sign_language_model.keras")

labels = [str(i) for i in range(model.output_shape[1])]


# ==========================================
# FRAME EXTRACTION
# ==========================================

def extract_frames(video_path, size=(224,224)):

    cap = cv2.VideoCapture(video_path)

    frames = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.resize(frame,size)
        frame = frame/255.0

        frames.append(frame)

    cap.release()

    if len(frames)==0:
        return None

    return np.mean(frames,axis=0)


# ==========================================
# PREDICTION FUNCTION
# ==========================================

def predict_sign(video):

    features = extract_frames(video)

    if features is None:
        return "Video could not be processed"

    features = np.expand_dims(features,axis=0)

    prediction = model.predict(features)

    index = np.argmax(prediction)

    confidence = np.max(prediction)*100

    result = labels[index]

    return f"Predicted Sign: {result} | Confidence: {confidence:.2f}%"


# ==========================================
# GRADIO GUI
# ==========================================

interface = gr.Interface(
    fn=predict_sign,
    inputs=gr.Video(label="Upload Sign Language Video"),
    outputs="text",
    title="🤟 Sign Language Recognition AI",
    description="Upload a sign language video and the AI model will detect the gesture"
)

interface.launch(share=True)